In [1]:
import datetime
import json
import ssl
import urllib.request

def main():
    print("===== 교량 안전점검 데이터 수집을 시작합니다 =====")
    try:
        start_year = int(input("시작 연도를 입력하세요: "))
        end_year = int(input("끝 연도를 입력하세요: "))
    except ValueError:
        print("연도는 숫자만 입력해야 합니다.")
        return

    # 💡 요약 데이터와 모든 컬럼이 담긴 전체 데이터(all_details)를 함께 받아옵니다.
    result_summary, all_details = getFacilStatsService(start_year, end_year)
    
    print("\n===== [최종 수집 결과 요약] =====")
    print(json.dumps(result_summary, indent=4, ensure_ascii=False))
    
    if all_details:
        print("💡 [팁] 사용할 수 있는 대표 컬럼명 안내:")
        print("   facilName(교량이름), juso(주소), roadNm(도로명), grade(안전등급),")
        print("   checkType(점검종류), openYear(준공일), length(연장), totWidth(총폭)")
        
        # 1. 조원에게 원하는 컬럼 입력받기
        user_input = input("조회하고 싶은 컬럼명을 입력하세요 (여러 개는 쉼표로 구분, 예: facilName, juso, grade)\n(엔터를 치면 건너뜁니다): ")
        
        selected_columns = [col.strip() for col in user_input.split(",") if col.strip()]
        
        if selected_columns:
            filtered_data = []
            
            for item in all_details:
                filtered_item = {col: item.get(col, "정보 없음") for col in selected_columns}
                filtered_data.append(filtered_item)
                
            # 4. 화면에 깔끔하게 출력 (데이터가 너무 많을 수 있으니 상위 5개만 샘플로 출력)
            print(f"\n===== [선택하신 컬럼 조회 결과 (상위 5개 샘플)] =====")
            print(json.dumps(filtered_data[:5], indent=4, ensure_ascii=False))
            print(f"▶ 총 {len(filtered_data)}개의 데이터가 필터링되었습니다.")
        else:
            print("\n▶ 컬럼 조회를 건너뜁니다.")
            
    else:
        print("\n▶ 수집된 상세 데이터가 없습니다.")
        
    checkFacil(result_summary)


def getRequestUrl(url):
    req = urllib.request.Request(url)
    try:
        context = ssl._create_unverified_context()
        response = urllib.request.urlopen(req, context=context)
        if response.getcode() == 200:
            print("[%s] Url Request Success" % datetime.datetime.now())
            return response.read().decode("utf-8")
    except Exception as e:
        print(e)
        print("[%s] Error for URL: %s" % (datetime.datetime.now(), url))
        return None

    
def getFacilList(h_year):
    service_url = "https://apis.data.go.kr/1613000/btiData/getBrdgList"
    parameters = (
        "?responseType=json&serviceKey="
        + "YOUR_DATA_GO_KR_KEY"
    )
    parameters += "&hyear=" + h_year

    url = service_url + parameters
    print(f"요청 URL: {url}\n")
    responseDecode = getRequestUrl(url)

    if responseDecode is None:
        return None
    else:
        try:
            return json.loads(responseDecode)
        except json.JSONDecodeError:
            print(f"[{h_year}년도] JSON 파싱 에러 발생")
            return None

    
def getFacilStatsService(nStartYear, nEndYear):
    jsonResult = []
    year_summary = {}

    for year in range(nStartYear, nEndYear + 1):
        h_year = f"{year}"
        jsonData = getFacilList(h_year)

        if jsonData and jsonData.get("response", {}).get("header", {}).get("resultMsg") == "NORMAL SERVICE":
            body = jsonData["response"].get("body", {})
            items_wrapper = body.get("items", "")
            
            year_summary[h_year] = {"name": [], "count": 0}
            
            if not items_wrapper or items_wrapper == "":
                print(f" {year}년도 안전점검 교량: 총 0개 수집 완료 (데이터 없음)")
                print("-" * 70)
                continue
                
            if isinstance(items_wrapper, list):
                Tname = items_wrapper
            else:
                Tname = [items_wrapper]

            for item in Tname:
                check_date = str(item.get("checkDate", ""))
                if check_date and check_date.startswith(h_year):
                    facilName = item.get("facilName", "").strip()
                    
                    year_summary[h_year]["name"].append(facilName)
                    year_summary[h_year]["count"] += 1
                    
                    # 💡 모든 컬럼을 통째로 보존합니다.
                    full_item = item.copy()
                    full_item["targetYear"] = h_year
                    jsonResult.append(full_item)
                    
            print(f" {year}년도 안전점검 교량: 총 {year_summary[h_year]['count']}개 수집 완료")
            print("-" * 70)
        else:
            year_summary[h_year] = {"name": [], "count": 0}
            print(f" {year}년도 안전점검 교량: 총 0개 수집 완료 (정상 응답 없음)")
            print("-" * 70)

    return year_summary, jsonResult


def checkFacil(result_summary):
    print("\n===== 점검 주기가 지난 교량 =====")
    latest = {}
    overdue_facil = []
    
    for year, data in result_summary.items():
        for name in data["name"]:
            latest[name] = int(year)
            
    now_year = datetime.datetime.now().year

    for name, year in latest.items():
        if (now_year - year) >= 2:
            overdue_facil.append(name)

    if overdue_facil:
        print(f"점검기간이 2년 이상 지난 교량이 총 {len(overdue_facil)}개 발견되었습니다.")
        print(f"샘플 목록(최대 5개): {overdue_facil[:5]}")
    else:
        print("점검기간 지난 교량이 없습니다.")
            

if __name__ == "__main__":
    main()

===== 교량 안전점검 데이터 수집을 시작합니다 =====
요청 URL: https://apis.data.go.kr/1613000/btiData/getBrdgList?responseType=json&serviceKey=YOUR_DATA_GO_KR_KEY&hyear=2024

[2026-05-27 10:52:06.254400] Url Request Success
 2024년도 안전점검 교량: 총 34740개 수집 완료
----------------------------------------------------------------------
요청 URL: https://apis.data.go.kr/1613000/btiData/getBrdgList?responseType=json&serviceKey=YOUR_DATA_GO_KR_KEY&hyear=2025

[2026-05-27 10:52:18.188902] Url Request Success
 2025년도 안전점검 교량: 총 35303개 수집 완료
----------------------------------------------------------------------
요청 URL: https://apis.data.go.kr/1613000/btiData/getBrdgList?responseType=json&serviceKey=YOUR_DATA_GO_KR_KEY&hyear=2026

[2026-05-27 10:52:28.240674] Url Request Success
 2026년도 안전점검 교량: 총 0개 수집 완료 (정상 응답 없음)
----------------------------------------------------------------------

===== [최종 수집 결과 요약] =====
{
    "2024": {
        "name": [
            "묘봉교(하)",
            "비룡1교",
            "원덕성교",
            "원천육교

In [13]:
import datetime
import json
import urllib.request

def main():
    print("===== 교량 안전점검 데이터 수집을 시작합니다 =====")
    try:
        start_year = int(input("시작 연도를 입력하세요: "))
        end_year = int(input("끝 연도를 입력하세요: "))
    except ValueError:
        print("연도는 숫자만 입력해야 합니다.")
        return

    print("\n--------------------------------------------------")
    print("1. 기본 요약 및 점검 지연 교량 확인 (빠름 ⚡)")
    print("2. 전체 데이터 수집 및 특정 컬럼 조회 (느림 🐢)")
    print("--------------------------------------------------")
    choice = input("원하시는 작업 번호를 선택하세요 (1 또는 2): ").strip()
    
    # 💡 2번을 선택했을 때만 상세 모드 스위치를 켭니다.
    is_detailed_mode = True if choice == '2' else False

    # 💡 스위치 상태를 함수로 같이 넘겨줍니다.
    result_summary, all_details = getFacilStatsService(start_year, end_year, is_detailed_mode)
    
    print("\n===== [최종 수집 결과 요약] =====")
    print(json.dumps(result_summary, indent=4, ensure_ascii=False))
    
    # 💡 상세 모드(2번)를 켰을 때만 컬럼 조회 기능을 실행합니다.
    if is_detailed_mode:
        if all_details:
            print("\n--------------------------------------------------")
            print("💡 [팁] 사용할 수 있는 대표 컬럼명 안내:")
            print("   facilName(교량이름), juso(주소), roadNm(도로명), grade(안전등급),")
            print("   checkType(점검종류), openYear(준공일), length(연장), totWidth(총폭)")
            print("--------------------------------------------------")
            
            user_input = input("조회하고 싶은 컬럼명을 입력하세요 (여러 개는 쉼표로 구분, 예: facilName, grade)\n(엔터를 치면 건너뜁니다): ")
            selected_columns = [col.strip() for col in user_input.split(",") if col.strip()]
            
            if selected_columns:
                filtered_data = []
                for item in all_details:
                    filtered_item = {col: item.get(col, "정보 없음") for col in selected_columns}
                    filtered_data.append(filtered_item)
                    
                print(f"\n===== [선택하신 컬럼 조회 결과 (상위 5개 샘플)] =====")
                print(json.dumps(filtered_data[:5], indent=4, ensure_ascii=False))
                print(f"▶ 총 {len(filtered_data)}개의 데이터가 필터링되었습니다.")
            else:
                print("\n▶ 컬럼 조회를 건너뜁니다.")
        else:
            print("\n▶ 수집된 상세 데이터가 없습니다.")
            
    checkFacil(result_summary)


def getRequestUrl(url):
    req = urllib.request.Request(url)
    try:
        context = ssl._create_unverified_context()
        response = urllib.request.urlopen(req, context=context)
        if response.getcode() == 200:
            print("[%s] Url Request Success" % datetime.datetime.now())
            return response.read().decode("utf-8")
    except Exception as e:
        print(e)
        print("[%s] Error for URL: %s" % (datetime.datetime.now(), url))
        return None

    
def getFacilList(h_year):
    service_url = "https://apis.data.go.kr/1613000/btiData/getBrdgList"
    parameters = (
        "?responseType=json&serviceKey="
        + "YOUR_DATA_GO_KR_KEY"
    )
    parameters += "&hyear=" + h_year

    url = service_url + parameters
    responseDecode = getRequestUrl(url)

    if responseDecode is None:
        return None
    else:
        try:
            return json.loads(responseDecode)
        except json.JSONDecodeError:
            print(f"[{h_year}년도] JSON 파싱 에러 발생")
            return None


# 💡 파라미터에 is_detailed_mode 스위치를 추가로 받습니다.
def getFacilStatsService(nStartYear, nEndYear, is_detailed_mode):
    jsonResult = []
    year_summary = {}

    for year in range(nStartYear, nEndYear + 1):
        h_year = f"{year}"
        jsonData = getFacilList(h_year)

        if jsonData and jsonData.get("response", {}).get("header", {}).get("resultMsg") == "NORMAL SERVICE":
            body = jsonData["response"].get("body", {})
            items_wrapper = body.get("items", "")
            
            year_summary[h_year] = {"name": [], "count": 0}
            
            if not items_wrapper or items_wrapper == "":
                print(f" {year}년도 안전점검 교량: 총 0개 수집 완료 (데이터 없음)")
                print("-" * 70)
                continue
                
            if isinstance(items_wrapper, list):
                Tname = items_wrapper
            else:
                Tname = [items_wrapper]

            for item in Tname:
                check_date = str(item.get("checkDate", ""))
                if check_date and check_date.startswith(h_year):
                    facilName = item.get("facilName", "").strip()
                    
                    year_summary[h_year]["name"].append(facilName)
                    year_summary[h_year]["count"] += 1
                    
                    # 💡 스위치가 켜져 있을 때(2번 선택 시)만 무거운 데이터를 리스트에 복사합니다.
                    if is_detailed_mode:
                        full_item = item.copy()
                        full_item["targetYear"] = h_year
                        jsonResult.append(full_item)
                    
            print(f" {year}년도 안전점검 교량: 총 {year_summary[h_year]['count']}개 수집 완료")
            print("-" * 70)
        else:
            year_summary[h_year] = {"name": [], "count": 0}
            print(f" {year}년도 안전점검 교량: 총 0개 수집 완료 (정상 응답 없음)")
            print("-" * 70)

    return year_summary, jsonResult


def checkFacil(result_summary):
    print("\n===== 점검 주기가 지난 교량 =====")
    latest = {}
    overdue_facil = []
    
    for year, data in result_summary.items():
        for name in data["name"]:
            latest[name] = int(year)
            
    now_year = datetime.datetime.now().year

    for name, year in latest.items():
        if (now_year - year) >= 2:
            overdue_facil.append(name)

    if overdue_facil:
        print(f"⚠ 점검기간이 2년 이상 지난 교량이 총 {len(overdue_facil)}개 발견되었습니다.")
        print(f"샘플 목록(최대 5개): {overdue_facil[:5]}")
    else:
        print("✔ 점검기간 지난 교량이 없습니다.")
            

if __name__ == "__main__":
    main()

===== 교량 안전점검 데이터 수집을 시작합니다 =====

--------------------------------------------------
1. 기본 요약 및 점검 지연 교량 확인 (빠름 ⚡)
2. 전체 데이터 수집 및 특정 컬럼 조회 (느림 🐢)
--------------------------------------------------
[2026-05-27 10:41:21.797997] Url Request Success


KeyboardInterrupt: 

In [3]:
import datetime
import json
import ssl
import urllib.request

def main():
    num = input("1번은 연도별 안전점검, 2번은 원하는 컬럼 조회")
    print("===== 교량 데이터 수집을 시작합니다 =====")
    try:
        start_year = int(input("시작 연도를 입력하세요: "))
        end_year = int(input("끝 연도를 입력하세요: "))
    except ValueError:
        print("연도는 숫자만 입력해야 합니다.")
        return

    if num == '1':
        result_summary, all_details = getFacilStatsService(start_year, end_year, opt = False)
    
        print("\n===== [최종 수집 결과 요약] =====")
        print(json.dumps(result_summary, indent=4, ensure_ascii=False))
        checkFacil(result_summary)
    elif num == '2':
        result_summary, all_details = getFacilStatsService(start_year, end_year, opt = True)
        if all_details:
            print(" 사용할 수 있는 대표 컬럼명 안내:")
            print(" facilName(교량이름), juso(주소), roadNm(도로명), grade(안전등급),")
            print(" checkType(점검종류), openYear(준공일), length(연장), totWidth(총폭)")
        
        user_input = input("컬럼명을 입력하세요. 여러개는 ,로 입력해주세요. 예시를 참고해주세요. ")
        
        selected_columns = [col.strip() for col in user_input.split(",") if col.strip()]
        data = []
        if selected_columns:
            
            for item in all_details:
                filtered_item = {col: item.get(col, "정보 없음") for col in selected_columns}
                data.append(filtered_item)
                
            json_filename = f"bridge_details_{start_year}_{end_year}.json"
            with open(json_filename, "w", encoding="utf-8") as f:
                json.dump(data, f, indent=4, ensure_ascii=False)
            print(f"▶ 총 {len(data)}개의 데이터가 저장되었습니다..")
        else:
            print("\n▶ 컬럼 조회를 건너뜁니다.")
            
    else:
        print("\n▶ 수집된 상세 데이터가 없습니다.")
        

def getRequestUrl(url):
    req = urllib.request.Request(url)
    try:
        response = urllib.request.urlopen(req)
        if response.getcode() == 200:
            print("[%s] Url Request Success" % datetime.datetime.now())
            return response.read().decode("utf-8")
    except Exception as e:
        print(e)
        print("[%s] Error for URL: %s" % (datetime.datetime.now(), url))
        return None

    
def getFacilList(h_year):
    service_url = "https://apis.data.go.kr/1613000/btiData/getBrdgList"
    parameters = (
        "?responseType=json&serviceKey="
        + "YOUR_DATA_GO_KR_KEY"
    )
    parameters += "&hyear=" + h_year

    url = service_url + parameters
    #중간에 오류나면 url 찍어보세요.
    # print(f"요청 URL: {url}\n")
    responseDecode = getRequestUrl(url)

    if responseDecode is None:
        return None
    else:
        try:
            return json.loads(responseDecode)
        except json.JSONDecodeError:
            print(f"[{h_year}년도] JSON 파싱 에러 발생")
            return None

    
def getFacilStatsService(nStartYear, nEndYear, opt):
    jsonResult = []
    year_summary = {}

    for year in range(nStartYear, nEndYear + 1):
        h_year = f"{year}"
        jsonData = getFacilList(h_year)
        #normal service가 데이터 제대로 들어왔을때 응답
        if jsonData and jsonData.get("response", {}).get("header", {}).get("resultMsg") == "NORMAL SERVICE":
            body = jsonData["response"].get("body", {})
            items_wrapper = body.get("items", "")
            #hyear가 기준년도 그 기준년도에 해당하는 교량 다 불러오기 
            year_summary[h_year] = {"name": [], "count": 0}
            #데이터가 아예 없을시
            if not items_wrapper or items_wrapper == "":
                print(f" {year}년도 교량: 총 0개 수집 완료 (데이터 없음)")
                print("-" * 70)
                continue
            #데이터가 1개일때 list 자체를 넘겨줘서 list일때랑 딕셔너리일때 구분
            if isinstance(items_wrapper, list):
                Tname = items_wrapper
            else:
                Tname = [items_wrapper]

            #checkDate가 최종안전점검일자
            for item in Tname:
                check_date = str(item.get("checkDate", ""))
                if check_date and check_date.startswith(h_year):
                    facilName = item.get("facilName", "").strip()

                    #year_summary 딕셔너리에 이름 저장하고 count수 늘리기
                    year_summary[h_year]["name"].append(facilName)
                    year_summary[h_year]["count"] += 1
                    
                    #원하는 컬럼 가져올때 전체 데이터가 필요한데 시간이 너무 오래걸려서
                    #필요할때 아니면 그냥 위에 year_summary만 넘겨주려고 따로뺐어요
                    if opt:
                        full_item = item.copy()
                        full_item["targetYear"] = h_year
                        jsonResult.append(full_item)
            if not opt:        
                print(f" {year}년도 안전점검 교량: 총 {year_summary[h_year]['count']}개 수집 완료")
                print("-" * 70)
        else:
            if not opt:
                year_summary[h_year] = {"name": [], "count": 0}
                print(f" {year}년도 안전점검 교량: 총 0개 수집 완료 (정상 응답 없음)")
                print("-" * 70)

    return year_summary, jsonResult


def checkFacil(result_summary):
    print("\n===== 점검 주기가 지난 교량 =====")
    latest = {}
    overdue_facil = []
    
    for year, data in result_summary.items():
        for name in data["name"]:
            latest[name] = int(year)
            
    now_year = datetime.datetime.now().year

    for name, year in latest.items():
        if (now_year - year) >= 2:
            overdue_facil.append(name)

    if overdue_facil:
        print(f"점검기간이 2년 이상 지난 교량이 총 {len(overdue_facil)}개 발견되었습니다.")
        print(f"샘플 목록(최대 5개): {overdue_facil[:5]}")
    else:
        print("점검기간 지난 교량이 없습니다.")
            

if __name__ == "__main__":
    main()

===== 교량 데이터 수집을 시작합니다 =====
[2026-05-27 10:56:31.090302] Url Request Success
[2026-05-27 10:56:43.491434] Url Request Success
[2026-05-27 10:56:48.553628] Url Request Success
 사용할 수 있는 대표 컬럼명 안내:
 facilName(교량이름), juso(주소), roadNm(도로명), grade(안전등급),
 checkType(점검종류), openYear(준공일), length(연장), totWidth(총폭)
▶ 총 70043개의 데이터가 저장되었습니다..


In [9]:
ServiceKey="YOUR_DATA_GO_KR_KEY"
import pandas as pd
import urllib
import datetime
import json

#[CODE 1]
def getRequestUrl(url):    
    req = urllib.request.Request(url)    
    try: 
        response = urllib.request.urlopen(req)
        if response.getcode() == 200:
            print ("[%s] Url Request Success" % datetime.datetime.now())
            return response.read().decode('utf-8')
    except Exception as e:
        print(e)
        print("[%s] Error for URL : %s" % (datetime.datetime.now(), url))
        return None



# [CODE 2] 게임물배급업 조회서비스에서 url을 만들기
# pageNo: 페이지번호
# numOfRows: 한 페이지 결과수(max: 100)
def getGamePublisherInfo(pageNo, numOfRows):
    service_url = "https://apis.data.go.kr/1741000/game_distributors/info"
    parameters = "?_type=json&serviceKey=" + ServiceKey   #인증키
    parameters += "&pageNo=" + str(pageNo)
    parameters += "&numOfRows=" + str(numOfRows)

    url = service_url + parameters
    print(f'요청 URL: {url}') #액세스 거부 여부 확인용 출력    
    responseDecode = getRequestUrl(url)   #[CODE 1]
    
    if (responseDecode == None):
        return None
    else:
         return json.loads(responseDecode)


#[CODE 3]
def getGamePublisherService(total_pages, rows_per_page):
    jsonResult = []
    csvresult = []
    
    for page in range(1, total_pages + 1):
        jsonData = getGamePublisherInfo(page, rows_per_page)

        # 1. jsonData가 있고, 그 안에 'response'가 있는지 확인
        if jsonData and 'response' in jsonData:
            body = jsonData['response'].get('body', {})
            items = body.get('items', {})
            
            # [핵심 수정] items 안에 'item'이 리스트 형태로 들어있는지 확인
            if items and 'item' in items:
                itemList = items['item']
                
                # 데이터가 1개일 때는 딕셔너리로, 여러 개일 때는 리스트로 올 수 있음 처리
                if isinstance(itemList, dict): # 1개일 때
                    itemList = [itemList]
                
                for item in itemList:
                    jsonResult.append(item)
                    csvresult.append(item)
                
                print(f"--- {page}페이지 수집 성공! ---")
            else:
                # 더 이상 가져올 'item'이 없는 경우 반복 중단
                print(f"{page}페이지에 더 이상 데이터가 없습니다.")
                break
                
    return jsonResult, csvresult


# [CODE 0] (메인함수)
def main():
    columns = []
    print("<< 게임물배급업 데이터를 수집합니다. >>")

    # 1. 수집 설정 (예: 1페이지부터 10개씩 돌리기)

    page_count = 1
    rows_per_page = 10

    # 2. 데이터 수집 실행시키기
    jsonResult, csvresult = getGamePublisherService(page_count, rows_per_page)

    if not jsonResult:  #URL 요청은 성공하였지만, 데이터 제공이 안된 경우??
        print("수집된 데이터가 없습니다. 인증키나 URL을 확인하세요")

    else:
        #파일저장 1 : json 파일
        with open('./{GamePublisherData.json', 'w', encoding='utf8') as outfile:
            jsonFile  = json.dumps(jsonResult, indent=4, sort_keys=True, ensure_ascii=False)
            outfile.write(jsonFile)
        #파일저장 2 : csv 파일
        
        if jsonResult:
            result_df = pd.DataFrame(jsonResult)
            result_df.to_csv('./GamePublisherData.csv', index=False, encoding='utf-8')
            print("데이터 저장 완료: .json, .csv")
        else:
            print("수집된 데이터가 없어 .csv 생성 불가")

if __name__ == '__main__':
    main()


<< 게임물배급업 데이터를 수집합니다. >>
요청 URL: https://apis.data.go.kr/1741000/game_distributors/info?_type=json&serviceKey=YOUR_DATA_GO_KR_KEY&pageNo=1&numOfRows=10
[2026-05-27 11:44:55.539763] Url Request Success
--- 1페이지 수집 성공! ---
데이터 저장 완료: .json, .csv


In [13]:
import datetime
import json
import urllib.request

def main():
    print("서울시 따릉이 데이터")
    
    # 1000개 긁어오기
    init_data = getdataList(1, 1000)
    
    if init_data and "stationInfo" in init_data:
        # list_total_count 활용해서 전체 데이터 긁어오기
        # total_count라서 int걸어주기
        total = int(init_data["stationInfo"].get("list_total_count", 0))
        print(f"전체 대여소 수 : {total}\n")
        
        all_stations = []
        
        #1000개씩 다 긁어오기
        for start_idx in range(1, total + 1, 1000):
            #1000으로하면 에러나서 999
            end_idx = min(start_idx + 999, total)
            print(f"데이터 수집 중... ({start_idx} ~ {end_idx})")
            
            json_data = getdataList(start_idx, end_idx)
            
            if json_data and "stationInfo" in json_data:
                station_data = json_data["stationInfo"].get("row", [])
                #append하면 데이터를 딕셔너리째로 저장해서 안예쁨
                all_stations.extend(station_data)
                
        # json 파일에 저장하기
        json_filename = "seoul_cycle_stations.json"
        with open(json_filename, "w", encoding="utf-8") as f:
            json.dump(all_stations, f, indent=4, ensure_ascii=False)
        
        print(f"▶ 총 {len(all_stations)}개의 대여소 정보: {json_filename}")
        
        # 4. 저장된 데이터가 어떻게 생겼는지 상위 2개만 미리보기로 출력
        if all_stations:
            print("\n예시 데이터 두개")
            print(json.dumps(all_stations[:2], indent=4, ensure_ascii=False))
            
    else:
        print("뭔가 오류가 있습니다")

def getRequestUrl(url):
    req = urllib.request.Request(url)
    try:
        response = urllib.request.urlopen(req)
        if response.getcode() == 200:
            print("[%s] Url Request Success" % datetime.datetime.now())
            return response.read().decode("utf-8")
    except Exception as e:
        print(e)
        print("[%s] Error for URL: %s" % (datetime.datetime.now(), url))
        return None
    

def getdataList(start_index = 1, end_index = 1000):
    base_url = "http://openapi.seoul.go.kr:8088"
    api_key = "YOUR_SEOUL_API_KEY"
    req_type = "json"
    service_name = "tbCycleStationInfo"
    url = f"{base_url}/{api_key}/{req_type}/{service_name}/{start_index}/{end_index}/"
    print(url)

    responseDecode = getRequestUrl(url)

    if responseDecode is None:
        return None
    else:
        try:
            return json.loads(responseDecode)
        except json.JSONDecodeError:
            print(f"[{start_index}~{end_index}] JSON 파싱 에러 발생")
            return None

if __name__ == '__main__':
    main()


서울시 따릉이 데이터
http://openapi.seoul.go.kr:8088/YOUR_SEOUL_API_KEY/json/tbCycleStationInfo/1/1000/
[2026-05-27 12:09:06.529420] Url Request Success
전체 대여소 수 : 3230

데이터 수집 중... (1 ~ 1000)
http://openapi.seoul.go.kr:8088/YOUR_SEOUL_API_KEY/json/tbCycleStationInfo/1/1000/
[2026-05-27 12:09:06.786799] Url Request Success
데이터 수집 중... (1001 ~ 2000)
http://openapi.seoul.go.kr:8088/YOUR_SEOUL_API_KEY/json/tbCycleStationInfo/1001/2000/
[2026-05-27 12:09:07.137630] Url Request Success
데이터 수집 중... (2001 ~ 3000)
http://openapi.seoul.go.kr:8088/YOUR_SEOUL_API_KEY/json/tbCycleStationInfo/2001/3000/
[2026-05-27 12:09:07.394519] Url Request Success
데이터 수집 중... (3001 ~ 3230)
http://openapi.seoul.go.kr:8088/YOUR_SEOUL_API_KEY/json/tbCycleStationInfo/3001/3230/
[2026-05-27 12:09:07.596287] Url Request Success
▶ 총 3230개의 대여소 정보: seoul_cycle_stations.json

예시 데이터 두개
[
    {
        "STA_LOC": "마포구",
        "RENT_ID": "ST-10",
        "RENT_NO": "00108",
        "RENT_NM": "서교동 사거리",
        "RENT_ID_NM": "10

In [ ]:
import datetime
import json
import urllib.request

def main():
    num = input("1번은 연도별 안전점검, 2번은 원하는 컬럼 조회")
    print("===== 교량 데이터 수집을 시작합니다 =====")
    try:
        start_year = int(input("시작 연도를 입력하세요: "))
        end_year = int(input("끝 연도를 입력하세요: "))
    except ValueError:
        print("연도는 숫자만 입력해야 합니다.")
        return

    if num == '1':
        result_summary, all_details = getFacilStatsService(start_year, end_year, opt = False)
    
        print("\n===== [최종 수집 결과 요약] =====")
        print(json.dumps(result_summary, indent=4, ensure_ascii=False))
        checkFacil(result_summary)
    elif num == '2':
        result_summary, all_details = getFacilStatsService(start_year, end_year, opt = True)
        if all_details:
            print(" 사용할 수 있는 대표 컬럼명 안내:")
            print(" facilName(교량이름), juso(주소), roadNm(도로명), grade(안전등급),")
            print(" checkType(점검종류), openYear(준공일), length(연장), totWidth(총폭)")
        
        user_input = input("컬럼명을 입력하세요. 여러개는 ,로 입력해주세요. 예시를 참고해주세요. ")
        
        #컬럼 입력받고 공백제거하기
        selected_columns = [col.strip() for col in user_input.split(",") if col.strip()]
        data = []
        if selected_columns:
            for item in all_details:
                #컬럼명:데이터 형식으로 딕셔너리로 저장
                filtered_item = {col: item.get(col, "정보 없음") for col in selected_columns}
                #리스트에 딕셔너리를 저장
                data.append(filtered_item)
            #데이터 json 파일로 저장하기
            json_filename = f"bridge_details_{start_year}_{end_year}.json"
            with open(json_filename, "w", encoding="utf-8") as f:
                json.dump(data, f, indent=4, ensure_ascii=False)

            print(f"▶ 총 {len(data)}개의 데이터가 저장되었습니다..")
        else:
            print("\n 컬럼 조회를 건너뜁니다.")
            
    else:
        print("\n▶ 수집된 상세 데이터가 없습니다.")
        

def getRequestUrl(url):
    req = urllib.request.Request(url)
    try:
        response = urllib.request.urlopen(req)
        if response.getcode() == 200:
            print("[%s] Url Request Success" % datetime.datetime.now())
            return response.read().decode("utf-8")
    except Exception as e:
        print(e)
        print("[%s] Error for URL: %s" % (datetime.datetime.now(), url))
        return None

    
def getFacilList(h_year):
    service_url = "https://apis.data.go.kr/1613000/btiData/getBrdgList"
    parameters = (
        "?responseType=json&serviceKey="
        + "YOUR_DATA_GO_KR_KEY"
    )
    parameters += "&hyear=" + h_year

    url = service_url + parameters
    #중간에 오류나면 url 찍어보세요.
    # print(f"요청 URL: {url}\n")
    responseDecode = getRequestUrl(url)

    if responseDecode is None:
        return None
    else:
        try:
            return json.loads(responseDecode)
        except json.JSONDecodeError:
            print(f"[{h_year}년도] JSON 파싱 에러 발생")
            return None

    
def getFacilStatsService(nStartYear, nEndYear, opt):
    jsonResult = []
    year_summary = {}

    for year in range(nStartYear, nEndYear + 1):
        h_year = f"{year}"
        jsonData = getFacilList(h_year)
        #normal service가 데이터 제대로 들어왔을때 응답
        if jsonData and jsonData.get("response", {}).get("header", {}).get("resultMsg") == "NORMAL SERVICE":
            body = jsonData["response"].get("body", {})
            items_wrapper = body.get("items", "")
            #hyear가 기준년도 그 기준년도에 해당하는 교량 다 불러오기 
            year_summary[h_year] = {"name": [], "count": 0}
            #데이터가 아예 없을시
            if not items_wrapper or items_wrapper == "":
                print(f" {year}년도 교량: 총 0개 수집 완료 (데이터 없음)")
                print("-" * 70)
                continue
            #데이터가 1개일때 list 자체를 넘겨줘서 list일때랑 딕셔너리일때 구분
            if isinstance(items_wrapper, list):
                Tname = items_wrapper
            else:
                Tname = [items_wrapper]

            #checkDate가 최종안전점검일자
            for item in Tname:
                check_date = str(item.get("checkDate", ""))
                if check_date and check_date.startswith(h_year):
                    facilName = item.get("facilName", "").strip()

                    #year_summary 딕셔너리에 이름 저장하고 count수 늘리기
                    year_summary[h_year]["name"].append(facilName)
                    year_summary[h_year]["count"] += 1
                    
                    #원하는 컬럼 가져올때 전체 데이터가 필요한데 시간이 너무 오래걸려서
                    #필요할때 아니면 그냥 위에 year_summary만 넘겨주려고 따로뺐어요
                    if opt:
                        full_item = item.copy()
                        full_item["targetYear"] = h_year
                        jsonResult.append(full_item)
            if not opt:        
                print(f" {year}년도 안전점검 교량: 총 {year_summary[h_year]['count']}개 수집 완료")
                print("-" * 70)
        else:
            if not opt:
                year_summary[h_year] = {"name": [], "count": 0}
                print(f" {year}년도 안전점검 교량: 총 0개 수집 완료 (정상 응답 없음)")
                print("-" * 70)

    return year_summary, jsonResult


def checkFacil(result_summary):
    print("\n===== 점검 주기가 지난 교량 =====")
    print("주기 설정시 2026년까지 검색한 경우에만 정확합니다")
    latest = {}
    overdue_facil = []
    #딕셔너리로 이름과 연도를 저장해서 중복의 경우 년도만 최신화
    for year, data in result_summary.items():
        for name in data["name"]:
            latest[name] = int(year)
            
    now_year = datetime.datetime.now().year

    #임의로 현재랑 2년이상 차이나는 교량 리스트에 저장
    for name, year in latest.items():
        if (now_year - year) >= 2:
            overdue_facil.append(name)
    #overdue 리스트에 값이 있을때 없을때 
    if overdue_facil:
        print(f"점검기간이 2년 이상 지난 교량이 총 {len(overdue_facil)}개 발견되었습니다.")
        print(f"샘플 목록(최대 5개): {overdue_facil[:5]}")
    else:
        print("점검기간 지난 교량이 없습니다.")
            

if __name__ == "__main__":
    main()